**Setup**

In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from tqdm import tqdm

In [5]:
M_xanthus = read_sbml_model("/home/mickael/github/M_xanthus-E_coli-Predation/M_xanthus_model_V3_hdca.xml")
M_xanthus

Name,myxo_model
Memory address,753bd05f3ad0
Number of metabolites,1224
Number of reactions,1339
Number of genes,1201
Number of groups,0
Objective expression,1.0*OF_BIOMASS - 1.0*OF_BIOMASS_reverse_80d2e
Compartments,"c, e"


In [6]:
#np.random.seed(1)

**Genetic Algorithm**

Create first individual

In [7]:
Exchange_list = []
for i in M_xanthus.exchanges._dict:
    Exchange_list.append(i)

n = 100
individual = [] # one individual = dict of EX reaction and their lower bound

for i in range(n):
    dico_temp = {}
    for j in Exchange_list:
        dico_temp[j] = np.random.randint(-1000,0)
    individual.append(dico_temp)

run the algorithm

In [8]:
generation = 0
final = 50

with tqdm(total=final) as pbar:
    while generation <= final:
        fitness = []
        for i in individual:
            for j in i:
                M_xanthus.reactions.get_by_id(j).lower_bound = i[j]
            FBA = M_xanthus.optimize()
            fitness.append(FBA.objective_value)

        sorted_list = fitness.copy()
        sorted_list.sort(reverse=True)

        best_index = []
        for i in range(int(30/100 * len(sorted_list))):
            best_index.append(fitness.index(sorted_list[i]))
            
        individual2 = []
        for b in best_index:
            individual2.append(individual[b])

        for i in range(n-30):
            dico_temp = {}
            for j in Exchange_list:
                dico_temp[j] = individual[np.random.choice(best_index)][j]
            
            individual2.append(dico_temp)

            mut = np.random.randint(0,100) # look for if there is a mutation
            if mut >= 95:
                if np.random.randint(1,3) == 1: # switch
                    choosen_reaction_1 = np.random.choice(Exchange_list)
                    choosen_reaction_2 = np.random.choice(Exchange_list)
                    store = individual2[i][choosen_reaction_1]

                    individual2[i][choosen_reaction_1] = individual2[i][choosen_reaction_2]
                    individual2[i][choosen_reaction_2] = store

                if np.random.randint(1,3) == 2: # change the value
                    choosen_reaction = np.random.choice(Exchange_list)
                    individual2[i][choosen_reaction] = np.random.randint(-1000, 0)
                
                if np.random.randint(1,3) == 3: # scramble
                    choosen_reaction_1 = np.random.choice(Exchange_list)
                    choosen_reaction_2 = np.random.choice(Exchange_list)
                    choosen_reaction_3 = np.random.choice(Exchange_list)
                    choosen_reaction_4 = np.random.choice(Exchange_list)
                    store = [individual2[i][choosen_reaction_1],individual2[i][choosen_reaction_2],individual2[i][choosen_reaction_3],individual2[i][choosen_reaction_4]]

                    individual2[i][choosen_reaction_1] = np.random.choice(store, replace = False)
                    individual2[i][choosen_reaction_2] = np.random.choice(store, replace = False)
                    individual2[i][choosen_reaction_3] = np.random.choice(store, replace = False)
                    individual2[i][choosen_reaction_4] = np.random.choice(store, replace = False)


        
        individual = individual2
        generation += 1
        pbar.update(1)

for i in range(len(individual)):
    print(str(fitness[i]) + ": " + str(individual[i]))

51it [02:14,  2.63s/it]                        

221.6862060481536: {'EX_malt_e': -803, 'EX_his_L_e': -581, 'EX_cd2_e': -235, 'EX_ferrich_e': -911, 'EX_tttnt_e': -456, 'EX_metox_e': -849, 'EX_btn_e': -908, 'EX_met_L_e': -977, 'EX_mg2_e': -587, 'EX_ac_e': -472, 'EX_arsenb_e': -321, 'EX_cbl1_e': -980, 'EX_cgly_e': -36, 'EX_fum_e': -625, 'EX_h2s_e': -462, 'EX_so4_e': -652, 'EX_pydx_e': -36, 'EX_spmd_e': -329, 'EX_ppi_e': -724, 'EX_mnl_e': -679, 'EX_pro_L_e': -73, 'EX_glyc3p_e': -481, 'EX_orn_e': -816, 'EX_salcn_e': -352, 'EX_pi_e': -846, 'EX_acgam_e': -761, 'EX_pyr_e': -827, 'EX_etoh_e': -897, 'EX_but_e': -859, 'EX_hom_L_e': -937, 'EX_na1_e': -325, 'EX_acald_e': -748, 'EX_mobd_e': -225, 'EX_metox_R_e': -496, 'EX_arbt_e': -434, 'EX_sucr_e': -751, 'EX_galt_e': -187, 'EX_glyb_e': -678, 'EX_leu_L_e': -859, 'EX_h2o_e': -791, 'EX_so3_e': -722, 'EX_h2_e': -476, 'EX_lys_L_e': -618, 'EX_k_e': -797, 'EX_n2o_e': -797, 'EX_cu2_e': -133, 'EX_Fe3_e': -890, 'EX_fru_e': -450, 'EX_gam_e': -848, 'EX_gly_cys_L_e': -632, 'EX_alaala_e': -686, 'EX_co2_e': -9

Separate steps

In [9]:
# fitness = []
# for i in individual:
#     for j in i:
#         M_xanthus.reactions.get_by_id(j).lower_bound = i[j]
#     FBA = M_xanthus.optimize()
#     fitness.append(FBA.objective_value)

# print(fitness)

In [10]:
# sorted_list = fitness.copy()
# sorted_list.sort(reverse=True)

# best_index = []

# for i in range(int(30/100 * len(sorted_list))):
#     best_index.append(fitness.index(sorted_list[i]))

# print(best_index)

In [11]:
# individual2 = []
# for b in best_index:
#     individual2.append(individual[b])

# for i in range(n-30):
#     dico_temp = {}
#     for j in Exchange_list:
#         dico_temp[j] = individual[np.random.choice(best_index)][j]
#     individual2.append(dico_temp)

# individual = individual2

In [12]:
# # mutation
# mut = np.random.randint(0,100)
# if mut >= 95:
#     choosen_ind = np.random.randint(0,len(individual2))
#     if np.random.randint(1,3) == 1: # switch
#         choosen_reaction_1 = np.random.choice(Exchange_list)
#         choosen_reaction_2 = np.random.choice(Exchange_list)
#         store = individual2[choosen_ind][choosen_reaction_1]

#         individual2[choosen_ind][choosen_reaction_1] = individual2[choosen_ind][choosen_reaction_2]
#         individual2[choosen_ind][choosen_reaction_2] = store

#     if np.random.randint(1,3) == 2: # change the value
#         choosen_reaction = np.random.choice(Exchange_list)
#         individual2[choosen_ind][choosen_reaction] = np.random.randint(-1000, 0)
    
#     if np.random.randint(1,3) == 3: # scramble
#         choosen_reaction_1 = np.random.choice(Exchange_list)
#         choosen_reaction_2 = np.random.choice(Exchange_list)
#         choosen_reaction_3 = np.random.choice(Exchange_list)
#         choosen_reaction_4 = np.random.choice(Exchange_list)
#         store = [individual2[choosen_ind][choosen_reaction_1],individual2[choosen_ind][choosen_reaction_2],individual2[choosen_ind][choosen_reaction_3],individual2[choosen_ind][choosen_reaction_4]]

#         individual2[choosen_ind][choosen_reaction_1] = np.random.choice(store, replace = False)
#         individual2[choosen_ind][choosen_reaction_2] = np.random.choice(store, replace = False)
#         individual2[choosen_ind][choosen_reaction_3] = np.random.choice(store, replace = False)
#         individual2[choosen_ind][choosen_reaction_4] = np.random.choice(store, replace = False)

